# HydroML — Exploratory Analysis Notebook
End-to-end demonstration of the drainage network extraction and flood risk pipeline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

from backend.utils.dem_processor import DEMProcessor
from backend.utils.geo_utils import (
    _synthetic_dem, generate_hillshade, apply_colormap
)
from backend.models.ensemble_model import EnsembleModel

print('All imports OK')

## 1. Generate / Load DEM

In [ ]:
# Generate a synthetic 256×256 DEM
dem = _synthetic_dem(256, 256)
print(f'DEM shape: {dem.shape}   range: {dem.min():.1f} – {dem.max():.1f} m')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dem, cmap='terrain')
plt.colorbar(im, ax=ax, label='Elevation (m)')
ax.set_title('Synthetic DEM')
plt.tight_layout()
plt.show()

## 2. Feature Extraction

In [ ]:
proc = DEMProcessor(cell_size=30.0)
features = proc.extract_all_features(dem)
print(f'Feature array shape: {features.shape}   (H, W, {features.shape[-1]} channels)')

feature_names = [
    'Elevation', 'Slope', 'Aspect', 'Plan Curv.', 'Profile Curv.',
    'Tex. Mean', 'Tex. Var.', 'Flow Acc.', 'TWI', 'SPI',
    'Ridge Dist.', 'Y coord', 'X coord'
]

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.ravel()
for i, name in enumerate(feature_names):
    ax = axes[i]
    im = ax.imshow(features[:, :, i], cmap='viridis')
    ax.set_title(name, fontsize=9)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
for ax in axes[len(feature_names):]:
    ax.axis('off')
plt.suptitle('Extracted Feature Maps', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Training Labels (D8 Flow Accumulation)

In [ ]:
labels = proc.generate_stream_labels(dem, flow_threshold=800)
stream_pct = labels.mean() * 100
print(f'Stream pixels: {labels.sum():,} ({stream_pct:.2f}%)')

hs = generate_hillshade(dem)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(hs, cmap='gray')
axes[0].set_title('Hillshade')
axes[0].axis('off')

axes[1].imshow(hs, cmap='gray', alpha=0.6)
masked = np.ma.masked_where(labels == 0, labels)
axes[1].imshow(masked, cmap='cool', alpha=0.85)
axes[1].set_title(f'D8 Stream Labels (threshold=800, {stream_pct:.1f}% streams)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. ML Model Training & Inference

In [ ]:
model = EnsembleModel(cell_size=30.0)
model.train(dem, flow_threshold=800)
print('Training complete.')

In [ ]:
results = model.predict(dem)

fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 4, figure=fig)

panels = [
    ('DEM',              dem,                        'terrain'),
    ('CNN Probability',  results['cnn_prob'],         'Blues'),
    ('XGBoost Prob.',    results['xgb_prob'],         'Purples'),
    ('Ensemble Prob.',   results['ensemble_prob'],    'plasma'),
    ('Hillshade',        hs.astype(float)/255,        'gray'),
    ('Stream Mask (ML)', results['stream_mask'],      'cool'),
    ('D8 Labels',        labels.astype(float),        'cool'),
]

for idx, (title, arr, cmap) in enumerate(panels):
    ax = fig.add_subplot(gs[idx // 4, idx % 4])
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Model Outputs', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Flood Risk Analysis

In [ ]:
for rainfall_mm in [25, 75, 150]:
    flood = model.compute_flood_risk(dem, results['ensemble_prob'], rainfall_mm)
    high  = (flood['risk_class'] == 3).mean() * 100
    med   = (flood['risk_class'] == 2).mean() * 100
    low   = (flood['risk_class'] == 1).mean() * 100
    print(f'Rainfall {rainfall_mm:3d} mm → High: {high:.1f}%  Med: {med:.1f}%  Low: {low:.1f}%')

# Final visualisation at 100 mm
flood = model.compute_flood_risk(dem, results['ensemble_prob'], 100)
cmap_risk = mcolors.LinearSegmentedColormap.from_list(
    'risk', ['#1b5e20', '#f9a825', '#b71c1c']
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(hs, cmap='gray')
axes[0].set_title('Hillshade Base')
axes[0].axis('off')

im = axes[1].imshow(flood['risk_index'], cmap=cmap_risk, vmin=0, vmax=1)
axes[1].set_title('Flood Risk Index')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], label='Risk [0–1]')

class_cmap = mcolors.ListedColormap(['#1b5e20', '#f9a825', '#b71c1c'])
im2 = axes[2].imshow(flood['risk_class'], cmap=class_cmap, vmin=1, vmax=3)
axes[2].set_title('Risk Classification (100 mm rainfall)')
axes[2].axis('off')
cbar = plt.colorbar(im2, ax=axes[2], ticks=[1.33, 2, 2.66])
cbar.set_ticklabels(['Low', 'Medium', 'High'])

plt.tight_layout()
plt.show()